# Notebook 01 — Data Loading & Database Setup

**Goal:** Load all 9 Olist CSV files into a SQLite database (`data/processed/sales_master.db`), create indexes on foreign keys for query performance, and validate row counts.

**Run this notebook first** before any other notebook in this project.

---

## Dataset Overview

The Brazilian E-Commerce Public Dataset by Olist contains real transaction data from 2016–2018 across the full order lifecycle:

| Table | Description | Expected Rows |
|---|---|---|
| `orders` | Order status & timestamps | ~99,441 |
| `customers` | Customer city/state | ~99,441 |
| `order_items` | Line items (price, freight) | ~112,650 |
| `products` | Product attributes | ~32,951 |
| `payments` | Payment type & value | ~103,886 |
| `reviews` | Review scores | ~99,224 |
| `sellers` | Seller locations | ~3,095 |
| `geolocation` | Zip → lat/long | ~1,000,163 |
| `category_translation` | Portuguese → English | ~71 |

In [ ]:
import sys
from pathlib import Path

# Make src/ importable from notebooks/
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text
import sqlite3

from src.db_setup import (
    RAW_DIR, PROCESSED_DIR, DB_PATH,
    CSV_TABLE_MAP, DATETIME_COLS, EXPECTED_ROWS,
    get_engine, load_csv, create_indexes,
    load_all_tables, validate_row_counts, save_kpi_summary
)

print(f'Project root : {project_root}')
print(f'Raw data dir : {RAW_DIR}')
print(f'Database path: {DB_PATH}')

## Step 1: Verify CSV Files Are Present

In [ ]:
print('Checking for CSV files in data/raw/:\n')
all_present = True
for csv_file in CSV_TABLE_MAP:
    path = RAW_DIR / csv_file
    exists = path.exists()
    size_mb = path.stat().st_size / 1_048_576 if exists else 0
    status = f'✓  {size_mb:6.1f} MB' if exists else '✗  MISSING'
    print(f'  {csv_file:<50} {status}')
    if not exists:
        all_present = False

print()
if all_present:
    print('All 9 CSV files found. Proceeding with database setup.')
else:
    print('ERROR: Missing files — place all CSVs in data/raw/ before continuing.')

## Step 2: Preview Each CSV

Quick sanity-check on column names and data types before loading into SQLite.

In [ ]:
for csv_file, table_name in CSV_TABLE_MAP.items():
    df = pd.read_csv(RAW_DIR / csv_file, nrows=3)
    print(f'\n--- {table_name} ({csv_file}) ---')
    print(f'  Columns: {list(df.columns)}')
    display(df.head(2))

## Step 3: Load All Tables into SQLite

Each CSV is read into a Pandas DataFrame and written to the SQLite database via SQLAlchemy's `to_sql()`. Tables are replaced if they already exist so this cell is idempotent.

In [ ]:
engine = get_engine()
print('Loading CSV files into SQLite...\n')
row_counts = load_all_tables(engine)
print('\nLoad complete.')

## Step 4: Create Indexes for Query Performance

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    create_indexes(conn)
print('Indexes created.')

## Step 5: Validate Row Counts

In [ ]:
passed = validate_row_counts(row_counts)
save_kpi_summary(row_counts)

total_records = sum(v for k, v in row_counts.items() if k not in ['geolocation'])
print(f'\nTotal records loaded (excl. geolocation): {total_records:,}')
print(f'Validation: {"ALL PASS" if passed else "SOME FAILURES — check above"}')

## Step 6: Quick Data Quality Check

Inspect null rates and date parsing for the orders table — the most critical table in the schema.

In [ ]:
orders = pd.read_sql_query('SELECT * FROM orders LIMIT 5000', engine)
print('Orders table — null counts per column:')
print(orders.isnull().sum().to_string())

print('\nSample order_purchase_timestamp values:')
print(orders['order_purchase_timestamp'].dropna().head(5).tolist())

In [ ]:
# Parse timestamps and check date range
orders['order_purchase_timestamp'] = pd.to_datetime(
    orders['order_purchase_timestamp'], errors='coerce'
)
print(f"Date range: {orders['order_purchase_timestamp'].min()} → {orders['order_purchase_timestamp'].max()}")
print(f"Null timestamps: {orders['order_purchase_timestamp'].isnull().sum()}")

# Order status distribution
print('\nOrder status distribution:')
print(orders['order_status'].value_counts().to_string())

## Step 7: Revenue Overview — Quick Bar Chart

A simple monthly revenue chart to confirm data loaded correctly before diving into advanced analysis.

In [ ]:
monthly_q = '''
SELECT
    strftime('%Y-%m', o.order_purchase_timestamp) AS month,
    ROUND(SUM(oi.price + oi.freight_value), 2)    AS revenue
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_purchase_timestamp IS NOT NULL
  AND o.order_status NOT IN ('canceled', 'unavailable')
  AND strftime('%Y', o.order_purchase_timestamp) IN ('2017', '2018')
GROUP BY month
ORDER BY month
'''
monthly = pd.read_sql_query(monthly_q, engine)

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(monthly['month'], monthly['revenue'] / 1000, color='steelblue', alpha=0.8)
ax.set_title('Monthly Revenue (BRL thousands) — 2017–2018', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (BRL thousands)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../reports/figures/monthly_revenue_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to reports/figures/')

## Summary

- All 9 CSV files loaded into `data/processed/sales_master.db`
- 11 performance indexes created on foreign key columns
- Row count validation passed for all tables
- Date range confirmed: dataset spans **September 2016 – August 2018**
- Monthly revenue chart confirms data integrity

**Next:** Run `02_advanced_sql_analysis.ipynb` to execute 15+ analytical SQL queries.